# Preparing the trial state

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we prepare a state that lets phase estimation return the ground-state energy reliably?

</div>

</div>

## Learning objectives

After completing this chapter, you will be able to:

- Explain why phase estimation requires an input state.
- Define state overlap and fidelity.
- Explain how ground-state fidelity measures target-state weight and influences phase-estimation outcomes.
- Construct a sparse trial wavefunction from important determinants.
- Generate a state-preparation logical circuit with the QDK/Chemistry sparse-isometry implementation.
- Distinguish trial-state quality from state-preparation logical circuit cost.

## Before you begin

This course requires a Python environment with the `qdk-chemistry[jupyter]` package.

`qdk-chemistry` ships compiled binaries for Linux, macOS on Apple silicon, and Windows on x86-64. This course also needs PySCF, which has no Windows build, so run it inside WSL on Windows. Run the cell below to check the current environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Setting up

The cell below imports the QDK/Chemistry pieces this chapter uses and quiets the solver logs.

In [ ]:
import json
from collections import Counter
from collections.abc import Iterator
from dataclasses import dataclass

import numpy as np
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import Circuit, Configuration, Hamiltonian, Wavefunction
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import Logger
from tutorial_choose_active_space import ActiveSpaceResult, run_active_space_workflow

@dataclass
class DeterminantContribution:
    """One determinant's contribution to the selected-space reference.

    Attributes:
        occupation: Spatial-orbital occupation string with storage padding removed.
        amplitude: CASCI coefficient after choosing the physically arbitrary
            global phase so the leading coefficient is positive real.
        weight: Squared coefficient magnitude, ``abs(amplitude)**2``.
        cumulative_weight: Sum of weights through this ranked determinant.
    """

    occupation: str
    amplitude: complex
    weight: float
    cumulative_weight: float

@dataclass
class TrialStateResult:
    r"""Quality and circuit cost for one determinant truncation.

    Fidelity is the squared overlap
    :math:`|\langle\Psi_{\mathrm{trial}}|\Psi_{\mathrm{reference}}\rangle|^2`.

    Attributes:
        num_determinants: Determinants retained by the projected calculation.
        trial_wavefunction: Normalized PMC wavefunction on the retained support.
        fidelity: Squared overlap with the selected-space CASCI ground state.
        circuit: Generated sparse-isometry state-preparation circuit.
        num_compute_qubits: Qubits in the occupation register.
        num_logical_gates: Decomposed leaf operations in the Q# circuit tree.
        logical_gate_counts: Childless-operation counts grouped by displayed gate name.
    """

    num_determinants: int
    trial_wavefunction: Wavefunction
    fidelity: float
    circuit: Circuit
    num_compute_qubits: int
    num_logical_gates: int
    logical_gate_counts: dict[str, int]

@dataclass
class TrialStateWorkflowResult:
    """Reference data and trial-state comparisons used by the chapter.

    Attributes:
        active_space_result: Coordinate-minimized selected molecular model.
        active_hamiltonian: Fermionic Hamiltonian in the selected orbital gauge.
        reference_determinants: Leading CASCI determinants for interpretation.
        trial_states: PMC/circuit results in requested determinant-count order.
    """

    active_space_result: ActiveSpaceResult
    active_hamiltonian: Hamiltonian
    reference_determinants: list[DeterminantContribution]
    trial_states: list[TrialStateResult]

from tutorial_prepare_trial_state import leading_determinants

## Workflow helpers

Two helpers follow: one ranks the reference determinants by weight, the other counts the leaf gates in a generated circuit. Run both so the rest of the notebook can use them.

In [ ]:
def leading_determinant_contributions(
    wavefunction: Wavefunction, max_determinants: int = 8
) -> list[DeterminantContribution]:
    """Summarize leading reference determinants and cumulative norm weight.

    Args:
        wavefunction: Selected-space CASCI reference.
        max_determinants: Number of ranked contributions to summarize.

    Returns:
        Ranked determinant records. Occupation strings are truncated to the
        physical active spatial-orbital count because ``Configuration`` storage
        can include trailing zero-valued capacity.
    """
    ranked_determinants = leading_determinants(wavefunction, max_determinants)
    leading_coefficient = complex(next(iter(ranked_determinants.values())))
    global_phase = leading_coefficient.conjugate() / abs(leading_coefficient)
    cumulative_weight = 0.0
    contributions = []
    alpha_channel = SymmetryLabel([axes.alpha()])
    num_active_spatial_orbitals = len(
        wavefunction.get_orbitals().active_indices().indices(alpha_channel)
    )
    for determinant, coefficient in ranked_determinants.items():
        # An eigenvector has arbitrary global phase. Make displayed amplitudes
        # reproducible without changing any weights, fidelities, or circuits.
        amplitude = complex(coefficient) * global_phase

        # Squared amplitudes contribute to the norm; their running sum shows how
        # much of the reference wavefunction the leading determinants capture.
        weight = float(abs(amplitude) ** 2)
        cumulative_weight += weight
        contributions.append(
            DeterminantContribution(
                # Configuration capacity may include zero-valued storage padding;
                # display only the physical selected active spatial orbitals.
                occupation=determinant.to_string()[:num_active_spatial_orbitals],
                amplitude=amplitude,
                weight=weight,
                cumulative_weight=cumulative_weight,
            )
        )
    return contributions

def iter_decomposed_gate_names(value: object) -> Iterator[str]:
    """Yield normalized names for childless operations in decomposed circuit JSON.

    Args:
        value: A dictionary, list, or scalar from the nested Q# circuit JSON tree.

    Yields:
        Gate names for records without nested child operations. A controlled X is
        displayed as ``CNOT`` for this tutorial's generated circuits.

    Notes:
        The generated circuits use one-control X operations. Production tooling
        should inspect the control count before generalizing this label to
        arbitrary multi-controlled X operations.
    """
    # The circuit is a nested tree of dictionaries and lists. yield from flattens
    # recursive results into one stream of decomposed gate names.
    if isinstance(value, dict):
        children = value.get("children")

        # Composite operations contain children; only gate records without
        # nested child operations are counted.
        if isinstance(children, list) and children:
            for child in children:
                yield from iter_decomposed_gate_names(child)
        else:
            gate_name = value.get("gate")
            if isinstance(gate_name, str):
                # Q# represents CNOT as an X gate with controls; distinguish it
                # from a bare X while the full gate record is available.
                yield (
                    "CNOT" if gate_name == "X" and value.get("controls") else gate_name
                )
        # The children branch was already traversed, so skip that key while
        # checking other fields for additional nested circuit structures.
        for key, child in value.items():
            if key != "children":
                yield from iter_decomposed_gate_names(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_decomposed_gate_names(child)


def circuit_statistics(circuit: Circuit) -> tuple[int, int, dict[str, int]]:
    """Count qubits and decomposed logical operations in a generated circuit.

    Args:
        circuit: State-preparation circuit whose Q# representation will be
            traversed recursively.

    Returns:
        A tuple containing the qubit count, total decomposed-gate count, and a
        deterministic gate-family count mapping.
    """
    # Convert the Q# circuit JSON to ordinary Python containers, then flatten
    # composite operations to the childless gate records that contribute to the count.
    circuit_data = json.loads(circuit.get_qsharp_circuit().json())
    logical_gate_names = list(iter_decomposed_gate_names(circuit_data))
    logical_gate_counts = Counter(logical_gate_names)
    return (
        len(circuit_data["qubits"]),
        len(logical_gate_names),
        dict(sorted(logical_gate_counts.items())),
    )

## Connection to the selected-space workflow

The selected-space CASCI calculation produced a normalized ground-state wavefunction spanning the $(n_\alpha,n_\beta)=(3,3)$ determinant sector introduced in the chapter *Mapping the problem to qubits*.
Each determinant represents one pattern of occupations among the selected active spin orbitals, and its coefficient is the corresponding amplitude in the wavefunction.
The *Jordan–Wigner encoding* represents the same occupation patterns on the compute register sized in the chapter *Mapping the problem to qubits*.

## Why phase estimation needs a trial state

Quantum phase estimation (QPE) estimates an eigenphase of a unitary operator.
For molecular energies, that unitary represents evolution under the Hamiltonian: each Hamiltonian eigenstate is also an eigenstate of the time-evolution operator, and its phase depends on its energy.
The phase-to-energy relationship and the QPE logical circuit are developed in the next chapter.
For now, the important point is that the compute register must contain a chosen quantum state before phase estimation can begin.

A state-preparation logical circuit initializes the compute register in this chosen normalized quantum state.
Here, *logical* means gates in the generated algorithmic circuit before error-correction code synthesis and hardware mapping.
This input is the trial state.
It is an approximation intended to contain a substantial contribution from the target ground state; QPE cannot begin from an unspecified state or create the ground state by searching through all possible wavefunctions.

The trial state can be written as a linear combination of the eigenstates $\{\vert\Psi_j\rangle\}$ of the active-space Hamiltonian:

$$
\vert\Psi_{\mathrm{trial}}\rangle
= \sum_j a_j\vert\Psi_j\rangle,
\qquad
\sum_j \left\vert a_j\right\vert^2=1.
$$

To isolate the effect of the input state, first assume that the requested trial state is prepared exactly, time evolution is exact, and phase readout has enough resolution to distinguish the relevant eigenphases.
Under these assumptions, an input eigenstate $\vert\Psi_j\rangle$ produces the phase corresponding to $E_j$.
For a trial state containing several eigenstates, a textbook coherent phase-estimation measurement samples the energy $E_j$ with probability $\left\vert a_j\right\vert^2$.
This probability statement assumes that one prepared system state produces one complete phase result.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why can phase estimation return an excited-state energy even when the ground-state energy is the target?</summary>

<div style="padding:0.1em 1em;">

A trial state can contain both ground- and excited-state eigenvectors.
Under the assumptions above, phase estimation returns each represented eigenvalue with probability equal to the squared magnitude of that eigenstate's amplitude in the trial state.

</div>

</details></div>

## Ground-state fidelity

As introduced in the tutorial overview, the ground-state fidelity is the squared overlap

$$
F
= \left\vert
    \langle\Psi_0\vert\Psi_{\mathrm{trial}}\rangle
  \right\vert^2.
$$

Both states are normalized, so $0\leq F\leq 1$.
The fidelity $F$ is the weight of the target ground state in the trial-state eigenstate expansion.
For the textbook coherent measurement described above, it is also the probability of sampling the ground-state eigenphase.

The QDK/Chemistry iterative quantum phase estimation (IQPE) implementation used later performs a different sampling procedure.
Each *phase bit* is one binary digit of the estimated phase fraction.
The implementation builds a separate circuit for each phase bit, and every circuit execution freshly prepares the trial state.
Each phase bit is selected by a majority vote over a specified number of circuit executions, then used as feedback for the next bit.
The final bit string therefore combines bitwise decisions from many state preparations rather than recording one eigenstate sample.

Fidelity remains a useful trial-state quality measure because it controls the ground-state contribution to those bit statistics.
However, it is not by itself the probability that one complete implemented IQPE run returns the ground-state energy, so it does not determine a trial count.
The complete result also depends on the other eigenstate weights and phases, the number of phase bits, shots per bit, phase feedback, and the Hamiltonian-simulation approximation.
The next chapter develops this bitwise sampling procedure and evaluates repeated complete IQPE runs.

Imperfect logical state preparation can also change the state actually loaded.
Residual logical faults after error correction can introduce further errors on a fault-tolerant machine, but they are not modeled by this tutorial's simulator.
These effects should be evaluated separately from the fidelity of the intended trial state.

## Running the reference workflow

Everything below builds on the selected active space from the previous chapter. This cell reruns that workflow, builds the active-space Hamiltonian, and ranks the leading reference determinants. It is the expensive step in the chapter.

In [ ]:
active_space_result = run_active_space_workflow()
reference_wavefunction = active_space_result.refined_casci_wavefunction
selected_orbitals = active_space_result.refined_orbitals
active_hamiltonian = create("hamiltonian_constructor", "qdk").run(
    selected_orbitals
)

reference_determinants = leading_determinant_contributions(
    reference_wavefunction
)
for contribution in reference_determinants[:4]:
    print(
        f"{contribution.occupation}  "
        f"amplitude {contribution.amplitude.real:+.4f}  "
        f"weight {contribution.weight:.4f}  "
        f"cumulative {contribution.cumulative_weight:.4f}"
    )

### Choosing the determinant count

The two cells that follow are one pass over one, two, and four determinants. Fix the count here so they run as ordinary cells.

In [ ]:
num_determinants = 4

## A sparse trial wavefunction

The selected-space CASCI wavefunction is classically tractable in this teaching example, so it provides a controlled reference for comparing trial states.
The script ranks its determinants by coefficient magnitude and retains the largest one, two, or four.
These determinant counts are examples rather than restrictions of the projected calculation or state-preparation method.
To compare other choices, change `determinant_counts` in the Jupyter notebook and rerun its cells.
In a larger problem where exact CASCI is unavailable, an approximate classical method must supply the candidate determinants and amplitudes for the trial state.

The script first prints the leading terms in the selected-space wavefunction.
Each occupation string contains one symbol for each selected active spatial orbital: `2` means doubly occupied, `u` means occupied by one $\alpha$ electron, `d` means occupied by one $\beta$ electron, and `0` means unoccupied.
The amplitude is the signed coefficient $c_I$ in $\vert\Psi_0\rangle=\sum_I c_I\vert\Phi_I\rangle$, while the weight $\left\vert c_I\right\vert^2$ is that determinant's contribution to the squared norm.
The cumulative weight shows how much of the norm is captured by the listed determinants.
The script computes these quantities directly from the leading CASCI coefficients.

Simply discarding coefficients and renormalizing would not optimize the wavefunction within the retained determinant space because the full-space amplitudes are not generally the amplitudes that minimize energy after determinants are removed.
A projected multi-configuration (PMC) calculation is a configuration-interaction calculation restricted to a user-specified set of determinants.
The QDK/Chemistry PMC calculator instead constructs the Hamiltonian matrix in the retained determinant space and solves its eigenvalue problem for the lowest-energy normalized eigenvector.
The resulting projected wavefunction has zero amplitude on every omitted determinant.
When using the projected wavefunction as a trial state, its overlap with the complete selected-space CASCI wavefunction therefore quantifies how much fidelity is retained after determinant truncation.

Under the *Jordan–Wigner encoding*, each retained determinant becomes one computational-basis state of the compute register.
The trial wavefunction is therefore

$$
\vert\Psi_{\mathrm{trial}}\rangle
=\sum_{I=1}^{K}\widetilde{c}_I\vert b_I\rangle,
$$

where $\vert b_I\rangle$ is the occupation bitstring for retained determinant $\Phi_I$, and $\widetilde{c}_I$ is its reoptimized amplitude.

The script constructs each projected trial state, forms the reference and trial coefficient vectors for the same retained determinants, and evaluates their squared inner product directly:

In [ ]:
# The leading reference determinants define the trial-state support;
# PMC then reoptimizes their amplitudes within that restricted space.
top_determinants = leading_determinants(
    reference_wavefunction, num_determinants
)
projected_calculator = create(
    "projected_multi_configuration_calculator", "macis_pmc"
)
_, trial_wavefunction = projected_calculator.run(
    active_hamiltonian, list(top_determinants)
)
retained_determinants = trial_wavefunction.get_active_determinants()

# Read both vectors in PMC determinant order. Without this alignment,
# np.vdot() could multiply coefficients belonging to different determinants.
reference_coefficients = np.asarray(
    [
        reference_wavefunction.get_coefficient(determinant)
        for determinant in retained_determinants
    ]
)
trial_coefficients = np.asarray(
    [
        trial_wavefunction.get_coefficient(determinant)
        for determinant in retained_determinants
    ]
)

# The trial vector is normalized on the retained support. The reference
# entries keep their full-state normalization, so their restricted norm
# records weight omitted by truncation.
fidelity = float(abs(np.vdot(reference_coefficients, trial_coefficients)) ** 2)

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why must fidelity be calculated separately from the PMC energy?</summary>

<div style="padding:0.1em 1em;">

The PMC calculation chooses the lowest-energy wavefunction within the retained determinant space, but it does not directly maximize overlap with the complete selected-space ground state.
Energy and overlap measure different properties, so the script evaluates fidelity explicitly.

</div>

</details></div>

## The trial state preparation logical circuit

The QDK/Chemistry sparse-isometry implementation converts the retained determinants into a binary matrix whose rows represent qubits and whose columns represent occupied-or-unoccupied patterns.
The method uses binary row operations to reduce the determinant patterns while recording controlled-NOT (CNOT) and X operations, prepares the reduced set of amplitudes, and reverses the recorded operations to expand the state across the compute register, in an optimized version of approaches introduced by Malvetti et al..
Students do not need to reproduce this synthesis by hand; it is implemented natively in QDK/Chemistry.
The important input is the normalized sparse wavefunction and the output is a logical circuit that prepares its amplitudes on the corresponding occupation states.

Some compute-register wires may have no gates in the state-preparation circuit.
The register begins in the all-zero occupation state, so a wire needs no preparation operation when the selected sparse wavefunction does not require that occupation bit to change or become entangled.
This does not make the qubit unnecessary: every compute qubit represents an active spin orbital on which the mapped active-space Hamiltonian acts.
The later controlled time evolution in QPE therefore requires the complete compute register and can couple the prepared determinant support to other configurations in the same sector.
Removing a gate-free preparation wire would change the Hamiltonian representation and the molecular problem, rather than merely simplify state preparation.

<div style="text-align:center;">

<svg xmlns="http://www.w3.org/2000/svg" width="1150" height="760" viewBox="0 0 1150 760" class="qdk-chemistry-state-preparation" role="img" data-asset="tutorial_qpe_state_preparation_comparison.svg" aria-label="Side-by-side logical state-preparation circuits on twelve compute qubits. The one-determinant circuit on the left contains six X gates that prepare one occupation bit string. The two-determinant circuit on the right contains rotations and controlled operations that prepare a coherent superposition of two occupation bit strings." style="max-width:100%;height:auto"> <style> .qdk-chemistry-state-preparation { --qdk-host-foreground: var(--vscode-editor-foreground, var(--jp-widgets-color, currentColor)); --circuit-fg: var(--qdk-host-foreground); --circuit-bg: transparent; --unitary-fill: transparent; --unitary-text: var(--qdk-host-foreground); --measure-fill: #0067b8; --measure-text: #ffffff; --ket-fill: #0067b8; --ket-text: #ffffff; } .qdk-chemistry-state-preparation line, .qdk-chemistry-state-preparation circle, .qdk-chemistry-state-preparation rect { stroke: var(--circuit-fg); stroke-width: 1; } .qdk-chemistry-state-preparation text { fill: var(--circuit-fg); dominant-baseline: middle; text-anchor: middle; font-family: "Segoe UI", Arial, sans-serif; } .qdk-chemistry-state-preparation .qs-mathtext { font-family: "Cambria Math", "STIX Two Math", serif; font-style: italic; } .qdk-chemistry-state-preparation .gate .qs-group-label { fill: var(--circuit-fg); text-anchor: start; } .qdk-chemistry-state-preparation .gate-unitary { fill: var(--unitary-fill); } .qdk-chemistry-state-preparation .gate text { fill: var(--unitary-text); } .qdk-chemistry-state-preparation .control-line, .qdk-chemistry-state-preparation .control-dot { fill: var(--circuit-fg); } .qdk-chemistry-state-preparation .oplus > line, .qdk-chemistry-state-preparation .oplus > circle { fill: var(--circuit-bg); stroke: var(--circuit-fg); stroke-width: 2; } .qdk-chemistry-state-preparation .gate-measure { fill: var(--measure-fill); } .qdk-chemistry-state-preparation .qs-line-measure, .qdk-chemistry-state-preparation .arc-measure { stroke: var(--measure-text); fill: none; } .qdk-chemistry-state-preparation .gate-ket { fill: var(--ket-fill); } .qdk-chemistry-state-preparation text.ket-text { fill: var(--ket-text); stroke: none; } .qdk-chemistry-state-preparation .register-classical { stroke-width: 0.5; } .qdk-chemistry-state-preparation .gate-collapse, .qdk-chemistry-state-preparation .gate-expand, .qdk-chemistry-state-preparation .dropzone-layer { display: none; } </style> <g transform="translate(0 0)"><g class="qubit-input-states"><text font-size="16" x="20" y="118" text-anchor="start" dominant-baseline="middle" data-wire="0" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">0</tspan>⟩</text><text font-size="16" x="20" y="170" text-anchor="start" dominant-baseline="middle" data-wire="1" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">1</tspan>⟩</text><text font-size="16" x="20" y="222" text-anchor="start" dominant-baseline="middle" data-wire="2" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">2</tspan>⟩</text><text font-size="16" x="20" y="274" text-anchor="start" dominant-baseline="middle" data-wire="3" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">3</tspan>⟩</text><text font-size="16" x="20" y="326" text-anchor="start" dominant-baseline="middle" data-wire="4" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">4</tspan>⟩</text><text font-size="16" x="20" y="378" text-anchor="start" dominant-baseline="middle" data-wire="5" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">5</tspan>⟩</text><text font-size="16" x="20" y="430" text-anchor="start" dominant-baseline="middle" data-wire="6" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">6</tspan>⟩</text><text font-size="16" x="20" y="482" text-anchor="start" dominant-baseline="middle" data-wire="7" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">7</tspan>⟩</text><text font-size="16" x="20" y="534" text-anchor="start" dominant-baseline="middle" data-wire="8" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">8</tspan>⟩</text><text font-size="16" x="20" y="606" text-anchor="start" dominant-baseline="middle" data-wire="9" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">9</tspan>⟩</text><text font-size="16" x="20" y="658" text-anchor="start" dominant-baseline="middle" data-wire="10" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">10</tspan>⟩</text><text font-size="16" x="20" y="710" text-anchor="start" dominant-baseline="middle" data-wire="11" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">11</tspan>⟩</text></g><g class="wires"><line x1="40" x2="332" y1="118" y2="118" class="qubit-wire"></line><line x1="40" x2="332" y1="170" y2="170" class="qubit-wire"></line><line x1="40" x2="332" y1="222" y2="222" class="qubit-wire"></line><line x1="40" x2="332" y1="274" y2="274" class="qubit-wire"></line><line x1="40" x2="332" y1="326" y2="326" class="qubit-wire"></line><line x1="40" x2="332" y1="378" y2="378" class="qubit-wire"></line><line x1="40" x2="332" y1="430" y2="430" class="qubit-wire"></line><line x1="40" x2="332" y1="482" y2="482" class="qubit-wire"></line><line x1="40" x2="332" y1="534" y2="534" class="qubit-wire"></line><line x1="40" x2="332" y1="606" y2="606" class="qubit-wire"></line><line x1="40" x2="332" y1="658" y2="658" class="qubit-wire"></line><line x1="40" x2="332" y1="710" y2="710" class="qubit-wire"></line></g><g><g class="gate" data-location="0,0" data-expanded="true"><rect class="gate-unitary" x="80" y="46" width="234" height="528" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0" data-expanded="true"><rect class="gate-unitary" x="90" y="72" width="208" height="492" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0-0,0"><g class="oplus" data-wire-ys="[118]" data-width="30"><circle cx="120" cy="118" r="15"></circle><line x1="120" x2="120" y1="103" y2="133"></line><line x1="105" x2="135" y1="118" y2="118"></line></g></g><g class="gate" data-location="0,0-0,0-0,1"><g class="oplus" data-wire-ys="[170]" data-width="30"><circle cx="120" cy="170" r="15"></circle><line x1="120" x2="120" y1="155" y2="185"></line><line x1="105" x2="135" y1="170" y2="170"></line></g></g><g class="gate" data-location="0,0-0,0-0,2"><g class="oplus" data-wire-ys="[222]" data-width="30"><circle cx="120" cy="222" r="15"></circle><line x1="120" x2="120" y1="207" y2="237"></line><line x1="105" x2="135" y1="222" y2="222"></line></g></g><g class="gate" data-location="0,0-0,0-0,3"><g class="oplus" data-wire-ys="[430]" data-width="30"><circle cx="120" cy="430" r="15"></circle><line x1="120" x2="120" y1="415" y2="445"></line><line x1="105" x2="135" y1="430" y2="430"></line></g></g><g class="gate" data-location="0,0-0,0-0,4"><g class="oplus" data-wire-ys="[482]" data-width="30"><circle cx="120" cy="482" r="15"></circle><line x1="120" x2="120" y1="467" y2="497"></line><line x1="105" x2="135" y1="482" y2="482"></line></g></g><g class="gate" data-location="0,0-0,0-0,5"><g class="oplus" data-wire-ys="[534]" data-width="30"><circle cx="120" cy="534" r="15"></circle><line x1="120" x2="120" y1="519" y2="549"></line><line x1="105" x2="135" y1="534" y2="534"></line></g></g></g><text font-size="14" x="100" y="87" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">PrepareSingleReferenceState</tspan></text><g class="gate-control gate-collapse"><circle cx="92" cy="74" r="10"></circle><path d="M85,74 h14"></path></g></g></g><text font-size="14" x="90" y="61" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">MakeSingleReferenceStateCircuit</tspan></text><g class="gate-control gate-collapse"><circle cx="82" cy="48" r="10"></circle><path d="M75,48 h14"></path></g></g></g></g> <g transform="translate(368 0)"><g class="qubit-input-states"><text font-size="16" x="20" y="118" text-anchor="start" dominant-baseline="middle" data-wire="0" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">0</tspan>⟩</text><text font-size="16" x="20" y="170" text-anchor="start" dominant-baseline="middle" data-wire="1" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">1</tspan>⟩</text><text font-size="16" x="20" y="222" text-anchor="start" dominant-baseline="middle" data-wire="2" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">2</tspan>⟩</text><text font-size="16" x="20" y="274" text-anchor="start" dominant-baseline="middle" data-wire="3" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">3</tspan>⟩</text><text font-size="16" x="20" y="326" text-anchor="start" dominant-baseline="middle" data-wire="4" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">4</tspan>⟩</text><text font-size="16" x="20" y="378" text-anchor="start" dominant-baseline="middle" data-wire="5" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">5</tspan>⟩</text><text font-size="16" x="20" y="430" text-anchor="start" dominant-baseline="middle" data-wire="6" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">6</tspan>⟩</text><text font-size="16" x="20" y="482" text-anchor="start" dominant-baseline="middle" data-wire="7" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">7</tspan>⟩</text><text font-size="16" x="20" y="534" text-anchor="start" dominant-baseline="middle" data-wire="8" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">8</tspan>⟩</text><text font-size="16" x="20" y="586" text-anchor="start" dominant-baseline="middle" data-wire="9" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">9</tspan>⟩</text><text font-size="16" x="20" y="658" text-anchor="start" dominant-baseline="middle" data-wire="10" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">10</tspan>⟩</text><text font-size="16" x="20" y="710" text-anchor="start" dominant-baseline="middle" data-wire="11" class="qs-maintext qs-qubit-label">|<tspan class="qs-mathtext">ψ</tspan><tspan baseline-shift="sub" font-size="65%">11</tspan>⟩</text></g><g class="wires"><line x1="40" x2="782" y1="118" y2="118" class="qubit-wire"></line><line x1="40" x2="782" y1="170" y2="170" class="qubit-wire"></line><line x1="40" x2="782" y1="222" y2="222" class="qubit-wire"></line><line x1="40" x2="782" y1="274" y2="274" class="qubit-wire"></line><line x1="40" x2="782" y1="326" y2="326" class="qubit-wire"></line><line x1="40" x2="782" y1="378" y2="378" class="qubit-wire"></line><line x1="40" x2="782" y1="430" y2="430" class="qubit-wire"></line><line x1="40" x2="782" y1="482" y2="482" class="qubit-wire"></line><line x1="40" x2="782" y1="534" y2="534" class="qubit-wire"></line><line x1="40" x2="782" y1="586" y2="586" class="qubit-wire"></line><line x1="40" x2="782" y1="658" y2="658" class="qubit-wire"></line><line x1="40" x2="782" y1="710" y2="710" class="qubit-wire"></line></g><g><g class="gate" data-location="0,0" data-expanded="true"><rect class="gate-unitary" x="80" y="46" width="684" height="580" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0" data-expanded="true"><rect class="gate-unitary" x="90" y="72" width="664" height="544" fill-opacity="0" stroke-dasharray="8, 8"></rect><g><g class="gate" data-location="0,0-0,0-0,0"><g><g><rect class="gate-unitary" x="100" y="150" width="40" height="40" data-wire-ys="[170]" data-width="40"></rect><text font-size="14" x="120" y="170" class="qs-maintext"><tspan class="qs-mathtext">S</tspan><tspan dx="2" dy="-3" style="font-size: 0.8em;">†</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-0,1"><g class="oplus" data-wire-ys="[274]" data-width="30"><circle cx="120" cy="274" r="15"></circle><line x1="120" x2="120" y1="259" y2="289"></line><line x1="105" x2="135" y1="274" y2="274"></line></g></g><g class="gate" data-location="0,0-0,0-0,2"><g class="oplus" data-wire-ys="[118]" data-width="30"><circle cx="120" cy="118" r="15"></circle><line x1="120" x2="120" y1="103" y2="133"></line><line x1="105" x2="135" y1="118" y2="118"></line></g></g><g class="gate" data-location="0,0-0,0-1,0"><g><g><rect class="gate-unitary" x="152" y="150" width="40" height="40" data-wire-ys="[170]" data-width="40"></rect><text font-size="14" x="172" y="170" class="qs-maintext"><tspan class="qs-mathtext">H</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-2,0"><g><g><rect class="gate-unitary" x="204" y="150" width="56" height="40" data-wire-ys="[170]" data-width="56"></rect><text font-size="14" x="232" y="163" class="qs-maintext"><tspan class="qs-mathtext">Rz</tspan></text><text font-size="12" x="232" y="178" class="arg-button">2.0269</text></g></g></g><g class="gate" data-location="0,0-0,0-3,0"><g><g><rect class="gate-unitary" x="272" y="150" width="40" height="40" data-wire-ys="[170]" data-width="40"></rect><text font-size="14" x="292" y="170" class="qs-maintext"><tspan class="qs-mathtext">H</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-4,0"><g><g><rect class="gate-unitary" x="324" y="150" width="40" height="40" data-wire-ys="[170]" data-width="40"></rect><text font-size="14" x="344" y="170" class="qs-maintext"><tspan class="qs-mathtext">S</tspan></text></g></g></g><g class="gate" data-location="0,0-0,0-5,0"><g><g><rect class="gate-unitary" x="376" y="150" width="56" height="40" data-wire-ys="[170]" data-width="56"></rect><text font-size="14" x="404" y="163" class="qs-maintext"><tspan class="qs-mathtext">Rz</tspan></text><text font-size="12" x="404" y="178" class="arg-button">3.1416</text></g></g></g><g class="gate" data-location="0,0-0,0-6,0"><line x1="464" x2="464" y1="170" y2="274" class="control-line" style="pointer-events: none;"></line><circle cx="464" cy="170" r="5" class="control-dot" data-wire-ys="[170]" data-width="10"></circle><g class="oplus" data-wire-ys="[274]" data-width="30"><circle cx="464" cy="274" r="15"></circle><line x1="464" x2="464" y1="259" y2="289"></line><line x1="449" x2="479" y1="274" y2="274"></line></g></g><g class="gate" data-location="0,0-0,0-7,0"><line x1="516" x2="516" y1="274" y2="586" class="control-line" style="pointer-events: none;"></line><circle cx="516" cy="274" r="5" class="control-dot" data-wire-ys="[274]" data-width="10"></circle><g class="oplus" data-wire-ys="[586]" data-width="30"><circle cx="516" cy="586" r="15"></circle><line x1="516" x2="516" y1="571" y2="601"></line><line x1="501" x2="531" y1="586" y2="586"></line></g></g><g class="gate" data-location="0,0-0,0-8,0"><line x1="568" x2="568" y1="170" y2="482" class="control-line" style="pointer-events: none;"></line><circle cx="568" cy="170" r="5" class="control-dot" data-wire-ys="[170]" data-width="10"></circle><g class="oplus" data-wire-ys="[482]" data-width="30"><circle cx="568" cy="482" r="15"></circle><line x1="568" x2="568" y1="467" y2="497"></line><line x1="553" x2="583" y1="482" y2="482"></line></g></g><g class="gate" data-location="0,0-0,0-9,0"><line x1="620" x2="620" y1="118" y2="534" class="control-line" style="pointer-events: none;"></line><circle cx="620" cy="118" r="5" class="control-dot" data-wire-ys="[118]" data-width="10"></circle><g class="oplus" data-wire-ys="[534]" data-width="30"><circle cx="620" cy="534" r="15"></circle><line x1="620" x2="620" y1="519" y2="549"></line><line x1="605" x2="635" y1="534" y2="534"></line></g></g><g class="gate" data-location="0,0-0,0-10,0"><line x1="672" x2="672" y1="118" y2="430" class="control-line" style="pointer-events: none;"></line><circle cx="672" cy="118" r="5" class="control-dot" data-wire-ys="[118]" data-width="10"></circle><g class="oplus" data-wire-ys="[430]" data-width="30"><circle cx="672" cy="430" r="15"></circle><line x1="672" x2="672" y1="415" y2="445"></line><line x1="657" x2="687" y1="430" y2="430"></line></g></g><g class="gate" data-location="0,0-0,0-11,0"><line x1="724" x2="724" y1="118" y2="222" class="control-line" style="pointer-events: none;"></line><circle cx="724" cy="118" r="5" class="control-dot" data-wire-ys="[118]" data-width="10"></circle><g class="oplus" data-wire-ys="[222]" data-width="30"><circle cx="724" cy="222" r="15"></circle><line x1="724" x2="724" y1="207" y2="237"></line><line x1="709" x2="739" y1="222" y2="222"></line></g></g></g><text font-size="14" x="100" y="87" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">StatePreparation</tspan></text><g class="gate-control gate-collapse"><circle cx="92" cy="74" r="10"></circle><path d="M85,74 h14"></path></g></g></g><text font-size="14" x="90" y="61" class="qs-maintext qs-group-label"><tspan class="qs-mathtext">MakeStatePreparationCircuit</tspan></text><g class="gate-control gate-collapse"><circle cx="82" cy="48" r="10"></circle><path d="M75,48 h14"></path></g></g></g></g> </svg>

*Generated logical state-preparation circuits for the one-determinant trial state (left) and two-determinant trial state (right). Both use the same twelve-qubit compute register; the additional operations prepare multiple amplitudes rather than additional spin orbitals.*

</div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does one-determinant state preparation look so different from multi-determinant preparation?</summary>

<div style="padding:0.1em 1em;">

One determinant is one occupation bit string and therefore one computational-basis state.
Starting from the all-zero state, X gates only need to flip the qubits representing occupied spin orbitals.
A multi-determinant wavefunction is instead a coherent superposition of distinct occupation bit strings.
Because X gates can only map one basis state to another, rotations are needed to create amplitudes, phase operations establish relative signs or phases, and entangling gates correlate occupation changes across qubits.
The exact gate sequence depends on the synthesis method, but the distinction between preparing one basis state and preparing a coherent superposition is general.

</div>

</details></div>

To measure the generated logical-circuit cost, the script traverses the decomposed *Q# circuit representation*, counts displayed gate records that have no nested child operations, and identifies controlled X gates as CNOT gates.
The script creates the QDK/Chemistry sparse-isometry implementation and inspects the generated Q# logical circuit.
The factory key sparse_isometry_gf2x is the implementation's current API identifier using a helper function to count gates:

In [ ]:
state_preparation = create("state_prep", "sparse_isometry_gf2x")
circuit = state_preparation.run(trial_wavefunction)
num_compute_qubits, num_logical_gates, logical_gate_counts = circuit_statistics(
    circuit
)

The reported *preparation logical gate count* is the number of these childless gate records in the generated Q# logical-circuit representation after the state-preparation operation has been decomposed.
This software-level logical gate count is not logical-circuit depth, a fault-tolerant resource estimate, or a physical-resource estimate.
It can change if the state-preparation or circuit-decomposition implementation changes; error correction affects downstream fault-tolerant and physical costs instead.

### Circuit cost

This cell prints the circuit cost for the trial state prepared above.

In [ ]:
print(f"Compute qubits: {num_compute_qubits}")
print(f"Preparation logical gate count: {num_logical_gates}")
print(f"Logical gate-family counts: {logical_gate_counts}")

## Trial-state quality and preparation cost

Trial-state truncation introduces a separate cost–quality tradeoff within the selected active space.
Retaining more determinants can improve fidelity with the selected-space ground state, but preparing more nonzero amplitudes generally requires more logical gates.
The relevant question is therefore how much fidelity is gained for each increase in state-preparation cost.

All three trial states describe the same selected active spin-orbital space, so they use the same compute register size.
Changing the number of retained determinants changes amplitudes and logical-circuit structure, not the number of spin orbitals represented.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Does retaining more determinants require more compute qubits?</summary>

<div style="padding:0.1em 1em;">

No.
Each trial state represents the same selected active spin-orbital space, so the compute-register size is unchanged.
The number of retained determinants affects state-preparation operations rather than the compute-register size.

</div>

</details></div>

## Comparing the three trial states

The script repeats the same construction for one, two, and four determinants. Repeat it here to see how fidelity responds to the retained determinant count.

In [ ]:
from tutorial_prepare_trial_state import run_trial_state_workflow

comparison = run_trial_state_workflow()
fidelities = {
    trial.num_determinants: trial.fidelity
    for trial in comparison.trial_states
}
for count, fidelity in fidelities.items():
    print(f"{count} determinants: fidelity {fidelity:.4f}")

## Running the preparation

The cells above have already run the preparation. Use their output to answer the questions below.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How do fidelity and preparation cost change as determinants are retained?</summary>

<div style="padding:0.1em 1em;">

The one-, two-, and four-determinant fidelities are approximately $0.4825$, $0.5864$, and $0.7324$, respectively.
Their generated logical circuits have preparation logical gate counts of 6, 14, and 30, respectively, while every logical circuit uses twelve compute qubits.
From one to two determinants, fidelity increases by approximately $0.104$ while the gate count increases by eight.
From two to four determinants, fidelity increases by approximately $0.146$ while the gate count increases by sixteen.
For these three generated circuits, the second expansion provides less fidelity gain per additional preparation gate than the first.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What does the single-determinant fidelity reveal about multireference character?</summary>

<div style="padding:0.1em 1em;">

Its fidelity is approximately $0.4825$, equal to the weight of the leading `222000` determinant.
No single determinant therefore carries a majority of the selected-space ground-state weight at this geometry.
The substantial weight distributed among additional determinants provides direct evidence of multireference character in the selected active-space orbital representation.

</div>

</details></div>

Explain what the leading determinant weight reveals about multireference character, and distinguish the fidelity improvement from the increased logical-circuit cost.
The final IQPE calculation uses the four-determinant trial state.

## Where truncation starts to pay off

A single determinant carries less than half the weight of the selected-space ground state. That is the direct evidence of multireference character, and it is why one determinant is not enough to start phase estimation from.

Complete `first_majority_count` so it returns the smallest determinant count in `fidelities` whose fidelity is greater than 0.5.

In [ ]:
from _unit import exercise


@exercise
def first_majority_count():
    return 1

**Hint**

`fidelities` maps each determinant count to its fidelity. You are asked for a count, not a fidelity, so you need the keys, filtered by what their values do. More than one count may qualify.

In [ ]:
@exercise
def first_majority_count():
    return min(c for c, f in fidelities.items() if f > 0.5)

Two determinants. The one-determinant fidelity is about 0.4825, so no single configuration holds a majority of the ground state at this stretched geometry.

## Further reading

- [State preparation ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/state_preparation.html)
- [Projected multi-configuration calculations ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/pmc.html)
- [Wavefunctions ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/data/wavefunction.html)
- [Quantum circuits ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/data/circuit.html)